In [4]:
import sys

import subprocess
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
import loompy
from pathlib import Path


In [6]:

H5AD   = Path('output/objects/hsc.h5ad')
DB_DIR = Path('data/scenic')
OUT    = Path('output/objects/scenic')
OUT.mkdir(parents=True, exist_ok=True)

TF_LIST = DB_DIR / 'mm_mgi_tfs.txt'
MOTIFS  = DB_DIR / 'motifs-v10nr_clust-nr.mgi-m0.001-o0.0.tbl'
DB_500  = DB_DIR / 'mm10_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather'
DB_10K  = DB_DIR / 'mm10_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather'

LOOM    = OUT / 'hsc_expr.loom'
ADJ     = OUT / 'adjacencies.csv'
REG     = OUT / 'regulons.csv'
AUC     = OUT / 'auc_mtx.csv'

N_JOBS = 8


In [11]:
LOOM

PosixPath('/project/output/scenic/hsc_expr.loom')

In [12]:
# Build the pyscenic command prefix.
# Tries: (1) --target bin, (2) PATH, (3) Python entry-point fallback.
import shutil as _shutil
_bin   = Path('/project/cache/python_pkgs/bin/pyscenic')
_which = _shutil.which('pyscenic')
if _bin.exists():
    PYSCENIC = [str(_bin)]
elif _which:
    PYSCENIC = [_which]
else:
    # pyscenic installed as a package but no bin script — invoke via entry point
    # -B: disable .pyc writing/reading so edits to cache/python_pkgs/*.py take effect immediately
    PYSCENIC = [sys.executable, '-B', '-c',
                'from pyscenic.cli.pyscenic import main; main()']


In [13]:
PYSCENIC

['/usr/bin/python3',
 '-B',
 '-c',
 'from pyscenic.cli.pyscenic import main; main()']

In [14]:
# ── 1. h5ad → loom ────────────────────────────────────────────────────────────
print("=== 1. Load h5ad and write loom ===")
adata = ad.read_h5ad(H5AD)
print(adata)

if LOOM.exists() and LOOM.stat().st_size > 0:
    print(f"  [SKIP] {LOOM} exists")
else:
    X = adata.X
    if sp.issparse(X):
        X = X.toarray()
    X = X.astype(np.float32).T          # loompy: genes × cells
    loompy.create(
        str(LOOM),
        layers   = {'': X},
        row_attrs = {'Gene':   np.array(adata.var_names)},
        col_attrs = {'CellID': np.array(adata.obs_names)},
    )
    print(f"  Written: {LOOM}  ({LOOM.stat().st_size / 1e9:.1f} GB)")

del adata   # free RAM before heavy steps


=== 1. Load h5ad and write loom ===
AnnData object with n_obs × n_vars = 50056 × 11057
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'cell.name', 'tube', 'sample.name', 'sample.name.long', 'compartment', 'nCount_HTO', 'nFeature_HTO', 'percent.mt', 'percent.rp', 'percent.hb', 'count_feat_ratio', 'percent.stress', 'S.Score', 'G2M.Score', 'Phase', 'dm2_max_prop', 'dm2_max_resid', 'dm2_class', 'pass_mt', 'pass_nFeature', 'pass_rp', 'pass_hb', 'pass_qc', 'pass_hto', 'scDblFinder.score', 'scDblFinder.class', 'pass_dbl', 'pass_cell', 'nCount_SCT', 'nFeature_SCT', 'sNN_pca.SCT_30_res.0.1', 'sNN_pca.SCT_30_res.0.3', 'sNN_pca.SCT_30_res.0.5', 'sNN_pca.SCT_30_res.0.7', 'sNN_pca.SCT_30_res.0.9', 'sNN_pca.SCT_30_res.1', 'sNN_pca.SCT_30_res.1.2', 'sNN_pca.SCT_30_res.1.3', 'sNN_pca.SCT_30_res.1.5', 'seurat_clusters', 'cytotrace2_score', 'cytotrace2_potency', 'cytotrace2_relative'
    obsm: 'X_pca', 'X_umap'
  [SKIP] /project/output/scenic/hsc_expr.loom exists


In [ ]:
# ── 2. GRN — direct Python call (bypasses dask/distributed entirely) ──────────
print("\n=== 2. GRN (grnboost2 via joblib) ===")
if ADJ.exists() and ADJ.stat().st_size > 0:
    print(f"  [SKIP] {ADJ} exists")
else:
    import loompy as lp
    from arboreto.algo import grnboost2

    with lp.connect(str(LOOM), mode='r', validate=False) as ds:
        expr_df = pd.DataFrame(
            ds[:, :].T,                        # cells × genes
            columns=ds.ra['Gene'],
            index=ds.ca['CellID'],
        )
    tf_names = pd.read_csv(TF_LIST, header=None)[0].tolist()
    print(f"  {expr_df.shape[0]:,} cells × {expr_df.shape[1]:,} genes | {len(tf_names):,} TFs")

    adjacencies = grnboost2(
        expression_data=expr_df,
        tf_names=tf_names,
        verbose=True,
        seed=42,
        n_jobs=N_JOBS,
    )
    adjacencies.to_csv(ADJ, index=False)
    print(f"  {len(adjacencies):,} links → {ADJ}")
    del expr_df


In [23]:
str(LOOM)

'/project/output/scenic/hsc_expr.loom'

In [18]:
    import loompy as lp
    from arboreto.algo import grnboost2

    with lp.connect(str(LOOM), mode='r', validate=False) as ds:
        expr_df = pd.DataFrame(
            ds[:, :].T,                        # cells × genes
            columns=ds.ra['Gene'],
            index=ds.ca['CellID'],
        )
    tf_names = pd.read_csv(TF_LIST, header=None)[0].tolist()
    print(f"  {expr_df.shape[0]:,} cells × {expr_df.shape[1]:,} genes | {len(tf_names):,} TFs")



  53,720 cells × 9,942 genes | 1,721 TFs


In [19]:
?grnboost2

Signature:
grnboost2(
    expression_data,
    gene_names=None,
    tf_names='all',
    client_or_address='local',
    early_stop_window_length=25,
    limit=None,
    seed=None,
    verbose=False,
    n_jobs=1,
)
Docstring:
Launch arboreto with [GRNBoost2] profile.

:param expression_data: one of:
       * a pandas DataFrame (rows=observations, columns=genes)
       * a dense 2D numpy.ndarray
       * a sparse scipy.sparse.csc_matrix
:param gene_names: optional list of gene names (strings). Required when a (dense or sparse) matrix is passed as
                   'expression_data' instead of a DataFrame.
:param tf_names: optional list of transcription factors. If None or 'all', the list of gene_names will be used.
:param client_or_address: one of:
       * None or 'local': a new Client(LocalCluster()) will be used to perform the computation.
       * string address: a new Client(address) will be used to perform the computation.
       * a Client instance: the specified Client instance 

In [20]:
expr_df


,Mrpl15,Lypla1,Tcea1,Atp6v1h,4732440D04Rik,Rb1cc1,St18,Pcmtd1,Rrs1,Vcpip1,...,mt-Atp6,mt-Co3,mt-Nd3,mt-Nd4l,mt-Nd4,mt-Nd5,mt-Nd6,mt-Cytb,ENSMUSG00000095041,DRE
Rx_3_AAACCAAAGAATCATC-1,0.000000,0.693147,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,1.386294,0.693147,...,3.332205,3.555348,0.693147,0.693147,2.397895,1.386294,0.000000,2.484907,1.098612,0.0
Rx_3_AAACCAAAGCATTGAT-1,0.000000,0.000000,0.693147,0.000000,0.693147,1.791759,0.0,0.693147,0.000000,0.000000,...,2.708050,2.772589,0.693147,0.000000,1.098612,1.098612,0.000000,1.791759,0.693147,0.0
Rx_3_AAACCAAAGCTATACG-1,1.098612,0.000000,1.098612,0.000000,0.000000,0.693147,0.0,0.693147,0.693147,0.693147,...,3.871201,3.583519,1.386294,0.693147,1.945910,1.945910,1.098612,2.302585,0.000000,0.0
Rx_3_AAACCAAAGTATGAAG-1,1.098612,0.000000,1.386294,0.000000,0.000000,1.098612,0.0,0.000000,0.000000,0.000000,...,3.332205,3.496508,1.386294,0.693147,2.079442,1.098612,0.693147,2.302585,0.000000,0.0
Rx_3_AAACCAAAGTCAATCG-1,0.693147,0.000000,1.386294,0.000000,0.693147,1.098612,0.0,0.000000,0.693147,0.000000,...,4.382027,3.806663,0.693147,1.791759,3.091043,2.079442,0.693147,3.178054,1.386294,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Rx_5_TGTGTTGAGCAGGGTA-1,1.386294,0.000000,1.791759,0.000000,0.000000,0.693147,0.0,0.000000,1.098612,0.693147,...,3.178054,3.433987,1.791759,1.098612,1.945910,1.609438,0.000000,1.791759,1.098612,0.0
Rx_5_TGTGTTGAGCGATAGG-1,0.000000,0.693147,0.000000,0.693147,0.000000,0.693147,0.0,0.000000,0.000000,0.000000,...,3.583519,3.713572,1.609438,1.386294,2.484907,2.397895,0.000000,2.708050,0.000000,0.0
Rx_5_TGTGTTGAGCTAGCAC-1,0.000000,0.000000,0.693147,0.693147,0.000000,0.000000,0.0,0.000000,0.693147,0.000000,...,3.931826,4.174387,1.609438,1.386294,2.995732,2.484907,0.693147,3.295837,0.000000,0.0
Rx_5_TGTGTTGAGGGTTGGA-1,0.693147,0.000000,1.386294,0.000000,0.000000,1.609438,0.0,0.000000,0.693147,0.000000,...,3.218876,3.178054,1.386294,0.693147,2.079442,1.945910,0.000000,1.791759,1.098612,0.0


In [21]:
d

NameError: name 'd' is not defined